In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd

# Read the dataset
df = pd.read_csv('/kaggle/input/q3-ka-ai-2026')

# Display first few rows
df.head()


In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check missing values per column
df.isnull().sum().sort_values(ascending=False)


In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

# Fill numerical missing values with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [ ]:
# Check number of duplicate rows
df.duplicated().sum()


In [ ]:
# Encode categorical variables if any exist
if len(cat_cols) > 0:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


In [ ]:
X = df.drop(columns=["target"])
y = df["target"]


In [ ]:
X = df.drop(columns=["target"])
y = df["target"]


In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np
f1_scores = []

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        eval_metric="F1",
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)

print(f"Average F1 Score across all folds: {np.mean(f1_scores):.4f}")


In [ ]:
from catboost import CatBoostClassifier

final_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    verbose=0,
    random_state=42
)

final_model.fit(X, y)


In [ ]:
import numpy as np
import pandas as pd
# Get feature importance values
importances = final_model.get_feature_importance()
feature_names = X.columns

# Create DataFrame for easy handling
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)





In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.barh(
    feature_importance_df["Feature"].head(20)[::-1],
    feature_importance_df["Importance"].head(20)[::-1]
)
plt.xlabel("Importance Score")
plt.title("Top 20 Feature Importances (CatBoost)")
plt.tight_layout()
plt.show()
golden_feature = feature_importance_df.iloc[0]

In [ ]:

print(" Golden Feature ")
print(f"Feature Name: {golden_feature['Feature']}")
print(f"Importance Score: {golden_feature['Importance']:.4f}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier


TARGET_COL = "target"
golden_feature = "P_2"

X_full = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_golden = df[[golden_feature]]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_accuracy(X, y, cv):
    scores = []
    for train_idx, val_idx in cv.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            eval_metric="Accuracy",
            verbose=0,
            random_seed=42
        )

        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        scores.append(accuracy_score(y_val, preds))

    return float(np.mean(scores)), scores


full_mean_acc, full_scores = cv_accuracy(X_full, y, skf)

gold_mean_acc, gold_scores = cv_accuracy(X_golden, y, skf)

print(f"Full model Accuracy (mean over folds):   {full_mean_acc:.5f} | folds={np.round(full_scores, 5)}")
print(f"Golden-only Accuracy (mean over folds):  {gold_mean_acc:.5f} | folds={np.round(gold_scores, 5)}")
print(f"Difference (Golden - Full):              {(gold_mean_acc - full_mean_acc):.5f}")
